In [7]:
%load_ext autoreload
%autoreload 2

In [1]:
import sys
from pathlib import Path

# add parent directory of the notebook to sys.path
parent_dir = Path.cwd().parent
sys.path.append(str(parent_dir))

In [2]:
import sqlite3
import csv
from config import Config
conn = sqlite3.connect(Config.BHAV_DB_FILE_PATH)

In [3]:
dropSignalTable = "drop table signal_runs"
conn.execute(dropSignalTable)
conn.commit()

In [4]:
import updateSignal
updateSignal.main()

Processing symbols: 100%|██████████| 3876/3876 [04:59<00:00, 12.96symbol/s]


In [ ]:
updateRank = """
    UPDATE bhavcopy
	SET rank = (
		SELECT CASE
			WHEN signal = -1 AND count > 25 THEN 0
			WHEN signal = -1 AND count > 15 THEN 1
			WHEN signal = -1 AND count > 5  THEN 2
			WHEN signal = 1  AND count > 25 THEN 6
			WHEN signal = 1  AND count > 15 THEN 5
			WHEN signal = 1  AND count > 5  THEN 4
			ELSE 3
		END
		FROM signal_runs r
		WHERE r.symbol = bhavcopy.symbol
		  AND r.trade_date = bhavcopy.trade_date
	)
"""
conn.execute(updateRank)
updateDefaultRank = "UPDATE bhavcopy SET rank = 4 where rank is null"
conn.execute(updateDefaultRank)
conn.commit()

# Input files for training needs to copied to the training folder

In [ ]:
query = """
    SELECT
        symbol,
        COUNT(trade_date) AS traded_days
    FROM bhavcopy
    WHERE trade_date >= strftime('%Y-%m-%d', 'now', 'start of day', '-12 months')
    AND symbol NOT LIKE '%etf'
    AND symbol NOT LIKE '%bees'
    GROUP BY symbol
    HAVING traded_days >= 150
    ORDER BY traded_days DESC;
"""
cur = conn.cursor()
cur.execute(query)
rows = cur.fetchall()

with open("./symbolmap.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["index", "symbol"])

    for idx, (symbol, traded_days) in enumerate(rows, start=1):
        writer.writerow([idx, symbol])



In [3]:
import pandas as pd
verificationDataQuery = "select * from bhavcopy where trade_date >= '2025-01-01'"
df = pd.read_sql(verificationDataQuery, conn)

# --- Optimize dtypes ---
df['symbol'] = df['symbol'].astype('category')
df['trade_date'] = pd.to_datetime(df['trade_date'], format='%Y-%m-%d')
df['close_price'] = df['close_price'].astype('float64')

# --- EMA calculations ---
g = df.groupby('symbol', observed=True)['close_price']
df['ema_10'] = (
    g.ewm(span=10, adjust=False)
        .mean()
        .reset_index(level=0, drop=True)
)

df['ema_21'] = (
    g.ewm(span=21, adjust=False)
        .mean()
        .reset_index(level=0, drop=True)
)

# --- Generate one CSV per month ---
df['year_month'] = df['trade_date'].dt.to_period('M')

for ym, monthly_df in df.groupby('year_month'):
    file_name = f"./testVerify_{ym.year}_{ym.month:02d}.csv"
    monthly_df.drop(columns='year_month').to_csv(file_name, index=False)

In [14]:
import numpy as np
FEATURES = [
    "prev_close_log",
    "open_price_log",
    "high_price_log",
    "low_price_log",
    "last_price_log",
    "close_price_log",
    "avg_price_log",
    "ttl_trd_qnty_log",
    "no_of_trades_log",
    "deliv_qty",
]
sql = """
WITH f AS (
    SELECT
        LN(1 + prev_close)   AS prev_close_log,
        LN(1 + open_price)   AS open_price_log,
        LN(1 + high_price)   AS high_price_log,
        LN(1 + low_price)    AS low_price_log,
        LN(1 + last_price)   AS last_price_log,
        LN(1 + close_price)  AS close_price_log,
        LN(1 + avg_price)    AS avg_price_log,
        LN(1 + ttl_trd_qnty) AS ttl_trd_qnty_log,
        LN(1 + no_of_trades) AS no_of_trades_log,
        deliv_qty
    FROM bhavcopy
    WHERE trade_date <= ?
)
SELECT
    /* means */
    AVG(prev_close_log),
    AVG(open_price_log),
    AVG(high_price_log),
    AVG(low_price_log),
    AVG(last_price_log),
    AVG(close_price_log),
    AVG(avg_price_log),
    AVG(ttl_trd_qnty_log),
    AVG(no_of_trades_log),
    AVG(deliv_qty),

    /* stddevs */
    sqrt(AVG(prev_close_log * prev_close_log)
         - AVG(prev_close_log) * AVG(prev_close_log)),
    sqrt(AVG(open_price_log * open_price_log)
         - AVG(open_price_log) * AVG(open_price_log)),
    sqrt(AVG(high_price_log * high_price_log)
         - AVG(high_price_log) * AVG(high_price_log)),
    sqrt(AVG(low_price_log * low_price_log)
         - AVG(low_price_log) * AVG(low_price_log)),
    sqrt(AVG(last_price_log * last_price_log)
         - AVG(last_price_log) * AVG(last_price_log)),
    sqrt(AVG(close_price_log * close_price_log)
         - AVG(close_price_log) * AVG(close_price_log)),
    sqrt(AVG(avg_price_log * avg_price_log)
         - AVG(avg_price_log) * AVG(avg_price_log)),
    sqrt(AVG(ttl_trd_qnty_log * ttl_trd_qnty_log)
         - AVG(ttl_trd_qnty_log) * AVG(ttl_trd_qnty_log)),
    sqrt(AVG(no_of_trades_log * no_of_trades_log)
         - AVG(no_of_trades_log) * AVG(no_of_trades_log)),
    sqrt(AVG(deliv_qty * deliv_qty)
         - AVG(deliv_qty) * AVG(deliv_qty))
FROM f;
"""
cur = conn.cursor()
cur.execute(sql, ("2024-12-31",))
row = cur.fetchone()
n = len(FEATURES)
means = np.array(row[:n], dtype=np.float32)
stds  = np.array(row[n:], dtype=np.float32)
stds = np.maximum(stds, 1e-8)

normalization_stats = {
    "mean": means,
    "std": stds,
}
np.save("./norm_stats.npy", normalization_stats)

In [ ]:
conn.close()